# V1-16: 상세 Prompt 기업명 검색어 확장 A/B 평가

15번의 `reasoning=False + schema grounding` 조건을 고정하고 Prompt만 비교한다. 모델·temperature·OpenDART 탐색·평가 사례·JSON schema는 모두 유지한다.

- **A. V15 기준 Prompt**: 기존의 역할·기본 보류 규칙
- **B. 상세 정책 Prompt**: 호출 조건, 검색 가설의 역할, 약칭/오타/한글-영문 표기 판단, 보류 기준, 금지 규칙, 출력 계약을 구체화

> Prompt에는 평가 정답 기업명이나 `하닉 → SK하이닉스`처럼 평가 사례의 직접 매핑을 넣지 않는다. 이 노트북은 개발 평가이며, 최종 선택 전에는 독립 holdout이 필요하다.

## 목적과 해석 기준

B가 A보다 Candidate Recall 또는 LLM 신규 기여를 높이면서, 구조화 출력 성공률·출력 계약 준수율·안전 보류 정확도를 유지하는지 확인한다. 모델의 지식 자체와 Prompt 효과는 완전히 분리할 수 없으므로, 결과는 이 로컬 모델·이 Prompt 조합의 비교로만 해석한다.

LLM은 검색어 후보만 제안한다. OpenDART 인덱스가 후보를 검증하며 LLM 제안만으로 `corp_code`를 자동 확정하지 않는다.

In [1]:
import io
from datetime import datetime, timezone
from importlib.metadata import version
import json
from math import ceil
import os
from pathlib import Path
import re
from statistics import median
import sys
import time
from typing import Literal
import xml.etree.ElementTree as ET
import zipfile

import pandas as pd
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from rapidfuzz import fuzz, process

/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')
for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
MODEL_NAME = os.environ.get('OLLAMA_MODEL', 'qwen3.6:27b')
TEMPERATURE = 0
REQUEST_TIMEOUT_S = 300
PROMPT_VERSION = 'entity-search-plan-v4-detailed-prompt-ab'

print(f'모델: {MODEL_NAME} / temperature={TEMPERATURE} / reasoning=False')
print(f'요청 timeout={REQUEST_TIMEOUT_S}초 / runner 옵션은 지정하지 않음')

모델: qwen3.6:27b / temperature=0 / reasoning=False
요청 timeout=300초 / runner 옵션은 지정하지 않음


In [3]:
sys.path.insert(0, str(project_root / 'src'))
from dart.client import CORP_CODE_URL, _get_bytes


def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(stream):
        raise RuntimeError(zip_bytes.decode('utf-8', errors='replace'))
    with zipfile.ZipFile(stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])
    root = ET.fromstring(xml_bytes)
    return [{child.tag: child.text or '' for child in item} for item in root.findall('list')]


corp_records = load_corp_records(api_key)
retrieved_at = datetime.now().astimezone().isoformat(timespec='seconds')
print(f'OpenDART 기업코드 조회 시각: {retrieved_at}')
print(f'전체 법인 레코드 수: {len(corp_records):,}')

OpenDART 기업코드 조회 시각: 2026-09-15T21:50:04+09:00
전체 법인 레코드 수: 119,281


In [4]:
def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())


def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(values.values()) for key, values in index.items()}


name_index = build_name_index(corp_records)
name_keys = list(name_index)
TARGETS = {
    '삼성전자': ('삼성전자', '005930'), 'SK하이닉스': ('SK하이닉스', '000660'),
    'LG전자': ('LG전자', '066570'), '현대자동차': ('현대자동차', '005380'),
    'NAVER': ('NAVER', '035420'), '카카오': ('카카오', '035720'),
    'POSCO홀딩스': ('POSCO홀딩스', '005490'), '삼성바이오로직스': ('삼성바이오로직스', '207940'),
    '셀트리온': ('셀트리온', '068270'), '기아': ('기아', '000270'),
}
target_records = {}
for key, (official_name, stock_code) in TARGETS.items():
    matches = [r for r in corp_records if normalize_name(r['corp_name']) == normalize_name(official_name) and r['stock_code'].strip() == stock_code]
    if len(matches) != 1:
        raise AssertionError(f'{key}: OpenDART 기준 법인 검증 실패 ({len(matches)}건)')
    target_records[key] = matches[0]
print(f'정규화 이름 키 수: {len(name_keys):,}')

정규화 이름 키 수: 219,600


In [5]:
CASES = [
    ('삼성전자 2024년 매출을 알려줘.', '삼성전자', ['삼성전자'], 'skip'),
    ('삼성 전자 실적을 요약해줘.', '삼성 전자', ['삼성전자'], 'skip'),
    ('삼전의 최근 매출 추세를 보여줘.', '삼전', ['삼성전자'], 'candidate'),
    ('삼성전쟈 영업이익을 알려줘.', '삼성전쟈', ['삼성전자'], 'candidate'),
    ('SK하이닉스의 사업보고서 기준 매출은?', 'SK하이닉스', ['SK하이닉스'], 'skip'),
    ('sk 하이닉스 실적을 알려줘.', 'sk 하이닉스', ['SK하이닉스'], 'skip'),
    ('하이닉스 매출 추세를 보고 싶어.', '하이닉스', ['SK하이닉스'], 'candidate'),
    ('하닉 실적을 정리해줘.', '하닉', ['SK하이닉스'], 'candidate'),
    ('LG전자 매출을 알려줘.', 'LG전자', ['LG전자'], 'skip'),
    ('엘지전자 영업이익 추이를 알려줘.', '엘지전자', ['LG전자'], 'candidate'),
    ('현대자동차의 최근 실적은?', '현대자동차', ['현대자동차'], 'skip'),
    ('현대차 매출을 알려줘.', '현대차', ['현대자동차'], 'candidate'),
    ('현차 최근 매출 추세는?', '현차', ['현대자동차'], 'candidate'),
    ('NAVER의 매출을 알려줘.', 'NAVER', ['NAVER'], 'skip'),
    ('네이버 영업이익을 알려줘.', '네이버', ['NAVER'], 'candidate'),
    ('카카오의 매출을 알려줘.', '카카오', ['카카오'], 'skip'),
    ('POSCO홀딩스 실적을 분석해줘.', 'POSCO홀딩스', ['POSCO홀딩스'], 'skip'),
    ('포스코홀딩스 영업이익을 보여줘.', '포스코홀딩스', ['POSCO홀딩스'], 'candidate'),
    ('삼바의 최근 실적은?', '삼바', ['삼성바이오로직스'], 'candidate'),
    ('셀트 매출 추세를 알려줘.', '셀트', ['셀트리온'], 'candidate'),
    ('기아차의 영업이익을 알려줘.', '기아차', ['기아'], 'candidate'),
    ('엘지 계열사의 실적을 알려줘.', '엘지', [], 'abstain'),
    ('현대 실적을 알려줘.', '현대', [], 'abstain'),
    ('감자우주물산2026의 매출을 알려줘.', '감자우주물산2026', [], 'abstain'),
    ('BLUEORBITVENTURES2049의 실적을 알려줘.', 'BLUEORBITVENTURES2049', [], 'abstain'),
    ('QXZ테스트법인9999 재무정보를 알려줘.', 'QXZ테스트법인9999', [], 'abstain'),
]
TEST_CASES = [{'질문': q, '원문 기업 표현': raw, '기대 대상': targets, 'LLM 기대 행동': action} for q, raw, targets, action in CASES]
print(f'전체 사례: {len(TEST_CASES)}개')

전체 사례: 26개


In [6]:
FUZZY_KEY_LIMIT, FUZZY_RESULT_LIMIT = 50, 5


def exact_records(query):
    return name_index.get(normalize_name(query), [])


def fuzzy_records(query):
    scored = process.extract(normalize_name(query), name_keys, scorer=fuzz.ratio, limit=FUZZY_KEY_LIMIT)
    best = {}
    for key, score, _ in scored:
        for record in name_index[key]:
            if record['corp_code'] not in best or score > best[record['corp_code']][1]:
                best[record['corp_code']] = (record, score)
    return sorted(best.values(), key=lambda item: (item[1], bool(item[0]['stock_code'].strip()), item[0]['modify_date']), reverse=True)[:FUZZY_RESULT_LIMIT]


def union_candidates(query, prefix):
    candidates = {}
    for record in exact_records(query):
        candidates[record['corp_code']] = {'record': record, 'sources': {f'{prefix}_exact'}}
    for record, _ in fuzzy_records(query):
        candidates.setdefault(record['corp_code'], {'record': record, 'sources': set()})['sources'].add(f'{prefix}_fuzzy')
    return candidates


def merge_candidates(*sets):
    merged = {}
    for candidate_set in sets:
        for corp_code, candidate in candidate_set.items():
            merged.setdefault(corp_code, {'record': candidate['record'], 'sources': set()})['sources'].update(candidate['sources'])
    return merged


def needs_llm(raw_name):
    return len(exact_records(raw_name)) != 1

print('LLM 호출 대상:', sum(needs_llm(case['원문 기업 표현']) for case in TEST_CASES))

LLM 호출 대상: 18


In [7]:
class EntitySearchPlan(BaseModel):
    decision: Literal['candidate', 'ambiguous', 'unknown'] = Field(
        description='검색 가설 제안, 포괄·다의적 표현, 또는 해석 불가 상태'
    )
    search_terms: list[str] = Field(
        default_factory=list,
        max_length=2,
        description='OpenDART에서 검증할 정식 기업명 검색 가설. corp_code·종목코드·설명 문장은 금지.'
    )

PROMPT_A = '''
당신은 한국 기업 리서치 시스템의 기업명 검색 계획기다.
프로그램은 원문 기업 표현을 OpenDART에서 고유하게 exact 확정하지 못했을 때만 당신을 호출한다.

원문이 명백한 약칭, 한글·영문 표기 차이, 또는 가벼운 철자 오타로 보이면 decision='candidate'와 함께
정식 기업명으로 보이는 검색 가설을 최대 2개 제안한다. 완전한 확신이 없어도 검색 가설은 제안할 수 있다.
이 가설은 정답 확정이 아니며, 뒤의 프로그램이 OpenDART 기업 목록으로 검증하고 필요하면 사용자에게 재질문한다.

원문이 포괄적인 그룹명, 여러 기업을 뜻할 수 있는 표현, 또는 실제 기업으로 해석할 근거가 없는 표현이면
decision='ambiguous' 또는 decision='unknown'을 반환하고 search_terms=[]로 둔다.

사용자가 언급하지 않은 계열사·경쟁사·관련 기업을 추가하지 않는다.
corp_code, 종목코드, 투자 의견, 설명 문장을 반환하지 않는다.
'''

PROMPT_B = '''
역할: 당신은 한국 기업 리서치 시스템의 'OpenDART 기업명 검색어 계획기'다. 기업을 확정하거나 corp_code를 고르는 역할이 아니다. 프로그램은 원문 기업 표현의 OpenDART exact 결과가 고유하지 않을 때만 호출한다.

목표: 사용자 질문의 원문 기업 표현을 OpenDART 기업 목록에서 다시 찾기 위한 검색 가설을 최대 2개 만든다. 제안은 후보 탐색 입력일 뿐이며, 뒤의 프로그램이 OpenDART 목록으로 존재 여부를 검증하고 필요하면 사용자에게 재질문한다.

candidate를 선택할 조건:
1. 원문이 특정 기업을 가리킨다고 볼 만한 널리 쓰이는 약칭·축약·브랜드식 표기인 경우, 그 기업의 정식 법인명 또는 공식 영문 표기를 검색 가설로 제안한다.
2. 띄어쓰기, 대소문자, 한글 음역/영문 공식 표기 차이, 한두 글자의 명백한 오타처럼 표면 형태만 달라도 특정 기업을 가리킬 근거가 충분한 경우, OpenDART에 있을 법한 표준 기업명으로 바꿔 제안한다.
3. 원문을 그대로 반복하는 것은 그 표현 자체가 정식 기업명 검색어로 유효할 가능성이 있을 때만 허용한다. 별칭·오타·음역 표기라면 같은 문자열을 반복하지 말고 표준 검색 가설을 제안한다.

ambiguous 또는 unknown을 선택할 조건:
1. 그룹명, 업종명, 보통명사, 한 단어가 여러 독립 기업을 가리킬 수 있는 표현처럼 특정 법인을 지목할 근거가 부족하면 ambiguous와 빈 목록을 반환한다.
2. 존재하지 않는 것으로 보이거나 임의 문자열이며 특정 기업으로 연결할 근거가 없으면 unknown과 빈 목록을 반환한다.

금지 규칙:
- 사용자가 언급하지 않은 계열사, 경쟁사, 유사 업종 회사를 연상으로 추가하지 않는다.
- 확신이 없다는 이유로 그럴듯한 실제 기업명을 임의로 만든다거나, raw 표현과 무관한 회사를 제안하지 않는다.
- corp_code, 종목코드, 거래소 코드, 투자 의견, 근거 설명, 문장을 search_terms에 넣지 않는다.
- 최종 선택·순위·확정이라는 표현을 하지 않는다.

출력 계약: decision은 candidate, ambiguous, unknown 중 하나다. candidate일 때만 공백을 제외한 기업명 검색어를 1~2개 반환한다. ambiguous와 unknown은 반드시 search_terms=[]를 반환한다. 설명 없이 JSON 객체만 반환한다.
'''

SCHEMA_JSON = json.dumps(EntitySearchPlan.model_json_schema(), ensure_ascii=False)
SYSTEM_PROMPTS = {
    'A. V15 기준 Prompt': PROMPT_A + '\n반드시 설명 없이 유효한 JSON 객체만 반환한다. 아래 JSON Schema를 만족한다.\nJSON Schema: ' + SCHEMA_JSON,
    'B. 상세 정책 Prompt': PROMPT_B + '\nJSON Schema: ' + SCHEMA_JSON,
}

planners = {}
for prompt_name, system_prompt in SYSTEM_PROMPTS.items():
    llm = ChatOllama(model=MODEL_NAME, base_url=OLLAMA_BASE_URL, temperature=TEMPERATURE, reasoning=False, client_kwargs={'timeout': REQUEST_TIMEOUT_S})
    planners[prompt_name] = llm.with_structured_output(EntitySearchPlan, include_raw=True)

RUN_CONFIG = {
    'run_at_utc': datetime.now(timezone.utc).isoformat(), 'model': MODEL_NAME, 'temperature': TEMPERATURE,
    'reasoning': False, 'schema_grounding': True, 'request_timeout_s': REQUEST_TIMEOUT_S,
    'prompt_version': PROMPT_VERSION, 'prompt_candidates': list(SYSTEM_PROMPTS),
    'langchain_version': version('langchain'), 'langchain_ollama_version': version('langchain-ollama'),
}
RUN_CONFIG

{'run_at_utc': '2026-09-15T12:50:12.070561+00:00',
 'model': 'qwen3.6:27b',
 'temperature': 0,
 'reasoning': False,
 'schema_grounding': True,
 'request_timeout_s': 300,
 'prompt_version': 'entity-search-plan-v4-detailed-prompt-ab',
 'prompt_candidates': ['A. V15 기준 Prompt', 'B. 상세 정책 Prompt'],
 'langchain_version': '1.2.10',
 'langchain_ollama_version': '1.1.0'}

In [8]:
def build_messages(prompt_name, question, raw_name):
    return [('system', SYSTEM_PROMPTS[prompt_name]), ('human', f'사용자 질문: {question}\n원문 기업 표현: {raw_name}')]


def clean_terms(plan):
    if plan is None or plan.decision != 'candidate':
        return [], list(plan.search_terms) if plan is not None else []
    terms, rejected, seen = [], [], set()
    for term in plan.search_terms:
        term, key = term.strip(), normalize_name(term)
        if not term or len(term) > 80 or not key or key.isdigit() or key in seen:
            rejected.append(term)
            continue
        seen.add(key)
        terms.append(term)
    return terms[:2], rejected


def follows_output_contract(plan, raw_terms):
    if plan is None:
        return False
    if plan.decision == 'candidate':
        return 1 <= len(raw_terms) <= 2 and all(term.strip() for term in raw_terms)
    return plan.decision in {'ambiguous', 'unknown'} and not raw_terms


for prompt_name in SYSTEM_PROMPTS:
    print(f'[사전 호출] {prompt_name} / 하닉 / 호출 시작', flush=True)
    started_at = time.perf_counter()
    try:
        result = planners[prompt_name].invoke(build_messages(prompt_name, '하닉 실적을 정리해줘.', '하닉'))
        print(f'[사전 호출] 완료 / {(time.perf_counter() - started_at) * 1_000:,.0f}ms / parsed={result["parsed"]} / error={result["parsing_error"]}', flush=True)
    except Exception as exc:
        print(f'[사전 호출] 실패 / {(time.perf_counter() - started_at) * 1_000:,.0f}ms / {type(exc).__name__}: {exc}', flush=True)

[사전 호출] A. V15 기준 Prompt / 하닉 / 호출 시작
[사전 호출] 완료 / 1,913ms / parsed=decision='candidate' search_terms=['하닉', '하닉홀딩스'] / error=None
[사전 호출] B. 상세 정책 Prompt / 하닉 / 호출 시작
[사전 호출] 완료 / 2,104ms / parsed=decision='candidate' search_terms=['하닉시스템', '하닉'] / error=None


## A/B 후보 실행

LLM 호출이 필요한 사례마다 A와 B를 모두 한 번씩 실행한다. 순서는 사례별로 A→B와 B→A를 교차해 한 후보가 항상 먼저 warm 상태를 받는 편향을 줄인다.

In [9]:
PROMPT_NAMES = list(SYSTEM_PROMPTS)
llm_case_count = sum(needs_llm(case['원문 기업 표현']) for case in TEST_CASES)
call_total, call_number, rows = llm_case_count * len(PROMPT_NAMES), 0, []
print(f'전체 평가 시작: LLM 호출 {call_total}회 ({llm_case_count}개 사례 × Prompt {len(PROMPT_NAMES)}개)', flush=True)

for case_number, case in enumerate(TEST_CASES, start=1):
    raw_name = case['원문 기업 표현']
    baseline = union_candidates(raw_name, 'raw')
    expected_codes = {target_records[name]['corp_code'] for name in case['기대 대상']}
    llm_called = needs_llm(raw_name)
    variants = {}
    order = PROMPT_NAMES if case_number % 2 else list(reversed(PROMPT_NAMES))

    for prompt_name in order:
        plan, error, raw_terms, terms, rejected, latency_ms = None, None, [], [], [], None
        llm_candidates = {}
        if llm_called:
            call_number += 1
            print(f'[{call_number}/{call_total}] {prompt_name} / {raw_name!r} / 호출 시작', flush=True)
            started_at = time.perf_counter()
            try:
                result = planners[prompt_name].invoke(build_messages(prompt_name, case['질문'], raw_name))
                plan, error = result['parsed'], result['parsing_error']
                raw_terms = list(plan.search_terms) if plan is not None else []
                terms, rejected = clean_terms(plan)
                llm_candidates = merge_candidates(*(union_candidates(term, 'llm') for term in terms))
            except Exception as exc:
                error = f'{type(exc).__name__}: {exc}'
            finally:
                latency_ms = (time.perf_counter() - started_at) * 1_000
                status = '완료' if error is None and plan is not None else '실패'
                print(f'[{call_number}/{call_total}] {status} / {latency_ms:,.0f}ms', flush=True)
        variants[prompt_name] = {
            'plan': plan, 'error': str(error) if error else None, 'raw_terms': raw_terms, 'terms': terms, 'rejected': rejected,
            'llm_candidates': llm_candidates, 'latency_ms': latency_ms,
        }

    for prompt_name in PROMPT_NAMES:
        result = variants[prompt_name]
        expanded = merge_candidates(baseline, result['llm_candidates'])
        plan = result['plan']
        raw_terms = result['raw_terms']
        term_validity = [bool(term.strip()) and bool(exact_records(term.strip())) for term in raw_terms]
        rows.append({
            **case, 'Prompt': prompt_name, 'LLM 호출': llm_called, 'LLM 결정': plan.decision if plan else None,
            'LLM 원본 검색어': raw_terms, '실제 탐색 검색어': result['terms'], '거부된 검색어': result['rejected'],
            '구조화 출력 성공': None if not llm_called else result['error'] is None and plan is not None,
            '출력 계약 준수': None if not llm_called else follows_output_contract(plan, raw_terms),
            '원본 검색어 exact 유효 여부': term_validity,
            '기준선 정답 포함': bool(expected_codes & set(baseline)) if expected_codes else None,
            'LLM 정답 포함': bool(expected_codes & set(result['llm_candidates'])) if expected_codes else None,
            '확장 후 정답 포함': bool(expected_codes & set(expanded)) if expected_codes else None,
            'LLM 신규 기여': bool((expected_codes & set(result['llm_candidates'])) - set(baseline)) if expected_codes else None,
            '안전 보류': plan is not None and plan.decision in {'ambiguous', 'unknown'} and not raw_terms if case['LLM 기대 행동'] == 'abstain' else None,
            'LLM 호출 시간(ms)': result['latency_ms'], '오류': result['error'],
        })

print(f'전체 평가 완료: 행 {len(rows)}개, LLM 호출 {call_number}회', flush=True)
len(rows)

전체 평가 시작: LLM 호출 36회 (18개 사례 × Prompt 2개)
[1/36] A. V15 기준 Prompt / '삼전' / 호출 시작
[1/36] 완료 / 1,911ms
[2/36] B. 상세 정책 Prompt / '삼전' / 호출 시작
[2/36] 완료 / 2,239ms
[3/36] B. 상세 정책 Prompt / '삼성전쟈' / 호출 시작
[3/36] 완료 / 1,474ms
[4/36] A. V15 기준 Prompt / '삼성전쟈' / 호출 시작
[4/36] 완료 / 1,634ms
[5/36] A. V15 기준 Prompt / '하이닉스' / 호출 시작
[5/36] 완료 / 1,943ms
[6/36] B. 상세 정책 Prompt / '하이닉스' / 호출 시작
[6/36] 완료 / 2,273ms
[7/36] B. 상세 정책 Prompt / '하닉' / 호출 시작
[7/36] 완료 / 1,668ms
[8/36] A. V15 기준 Prompt / '하닉' / 호출 시작
[8/36] 완료 / 1,940ms
[9/36] B. 상세 정책 Prompt / '엘지전자' / 호출 시작
[9/36] 완료 / 2,197ms
[10/36] A. V15 기준 Prompt / '엘지전자' / 호출 시작
[10/36] 완료 / 1,835ms
[11/36] B. 상세 정책 Prompt / '현대차' / 호출 시작
[11/36] 완료 / 2,245ms
[12/36] A. V15 기준 Prompt / '현대차' / 호출 시작
[12/36] 완료 / 1,890ms
[13/36] A. V15 기준 Prompt / '현차' / 호출 시작
[13/36] 완료 / 1,842ms
[14/36] B. 상세 정책 Prompt / '현차' / 호출 시작
[14/36] 완료 / 1,657ms
[15/36] A. V15 기준 Prompt / '네이버' / 호출 시작
[15/36] 완료 / 1,640ms
[16/36] B. 상세 정책 Prompt / '네이버' / 호출 시작
[16/36] 완료 / 

52

In [10]:
def p95(values):
    return sorted(values)[ceil(len(values) * 0.95) - 1] if values else None


summary_rows = []
for prompt_name in PROMPT_NAMES:
    prompt_rows = [row for row in rows if row['Prompt'] == prompt_name]
    positive = [row for row in prompt_rows if row['기대 대상']]
    candidate = [row for row in prompt_rows if row['LLM 기대 행동'] == 'candidate' and row['LLM 호출']]
    abstain = [row for row in prompt_rows if row['LLM 기대 행동'] == 'abstain']
    llm_rows = [row for row in prompt_rows if row['LLM 호출']]
    validity = [value for row in llm_rows for value in row['원본 검색어 exact 유효 여부']]
    latencies = [row['LLM 호출 시간(ms)'] for row in llm_rows if row['LLM 호출 시간(ms)'] is not None]
    summary_rows.append({
        'Prompt': prompt_name,
        'Candidate Recall': sum(row['확장 후 정답 포함'] for row in positive) / len(positive),
        'LLM Candidate Recall': sum(row['LLM 정답 포함'] for row in candidate) / len(candidate),
        'LLM Expansion Contribution': sum(row['LLM 신규 기여'] for row in positive) / len(positive),
        'Abstention Accuracy': sum(row['안전 보류'] for row in abstain) / len(abstain),
        'Structured-output Success Rate': sum(row['구조화 출력 성공'] for row in llm_rows) / len(llm_rows),
        'Output Contract Validity': sum(row['출력 계약 준수'] for row in llm_rows) / len(llm_rows),
        'OpenDART Exact Term Validity': sum(validity) / len(validity) if validity else None,
        'LLM Call Rate': len(llm_rows) / len(prompt_rows),
        'p50 ms': median(latencies) if latencies else None, 'p95 ms': p95(latencies),
    })

summary = pd.DataFrame(summary_rows).set_index('Prompt')
display(summary)

,Candidate Recall,LLM Candidate Recall,LLM Expansion Contribution,Abstention Accuracy,Structured-output Success Rate,Output Contract Validity,OpenDART Exact Term Validity,LLM Call Rate,p50 ms,p95 ms
Prompt,,,,,,,,,,
A. V15 기준 Prompt,0.809524,0.666667,0.047619,1.0,1.0,1.0,0.47619,0.692308,1777.293353,2045.563432
B. 상세 정책 Prompt,0.857143,0.666667,0.095238,1.0,1.0,1.0,0.55000,0.692308,1671.629097,2273.497622


In [11]:
columns = [
    'Prompt', '원문 기업 표현', 'LLM 기대 행동', 'LLM 결정', 'LLM 원본 검색어', '실제 탐색 검색어',
    '기준선 정답 포함', 'LLM 정답 포함', '확장 후 정답 포함', 'LLM 신규 기여',
    '안전 보류', '출력 계약 준수', '원본 검색어 exact 유효 여부', 'LLM 호출 시간(ms)', '오류',
]
display(pd.DataFrame(rows)[columns].sort_values(['원문 기업 표현', 'Prompt']))

,Prompt,원문 기업 표현,LLM 기대 행동,LLM 결정,LLM 원본 검색어,실제 탐색 검색어,기준선 정답 포함,LLM 정답 포함,확장 후 정답 포함,LLM 신규 기여,안전 보류,출력 계약 준수,원본 검색어 exact 유효 여부,LLM 호출 시간(ms),오류
48,A. V15 기준 Prompt,BLUEORBITVENTURES2049,abstain,unknown,[],[],None,None,None,None,True,True,[],1318.666727,None
49,B. 상세 정책 Prompt,BLUEORBITVENTURES2049,abstain,unknown,[],[],None,None,None,None,True,True,[],1675.099925,None
16,A. V15 기준 Prompt,LG전자,skip,NaN,[],[],True,False,True,False,None,None,[],NaN,None
17,B. 상세 정책 Prompt,LG전자,skip,NaN,[],[],True,False,True,False,None,None,[],NaN,None
26,A. V15 기준 Prompt,NAVER,skip,NaN,[],[],True,False,True,False,None,None,[],NaN,None
27,B. 상세 정책 Prompt,NAVER,skip,NaN,[],[],True,False,True,False,None,None,[],NaN,None
32,A. V15 기준 Prompt,POSCO홀딩스,skip,NaN,[],[],True,False,True,False,None,None,[],NaN,None
33,B. 상세 정책 Prompt,POSCO홀딩스,skip,NaN,[],[],True,False,True,False,None,None,[],NaN,None
50,A. V15 기준 Prompt,QXZ테스트법인9999,abstain,unknown,[],[],None,None,None,None,True,True,[],1344.048142,None
51,B. 상세 정책 Prompt,QXZ테스트법인9999,abstain,unknown,[],[],None,None,None,None,True,True,[],1108.137857,None


## 채택 기준

- B가 A보다 Candidate Recall 또는 LLM Expansion Contribution을 높이고, Abstention Accuracy·Structured-output Success Rate·Output Contract Validity를 낮추지 않으면 B를 다음 모델 비교의 Prompt 후보로 조건부 채택한다.
- B가 후보 회수만 높이지만 모호·가상 입력에 실제 기업을 주입하면 미채택한다.
- 두 Prompt가 비슷하면 품질만으로 우열을 단정하지 않고, 독립 holdout 또는 API 모델 비교로 보류한다.
- 지연시간은 순서를 교차했지만 단일 실행 결과이므로 참고 지표다. timeout·파싱 실패는 운영 안정성 실패로 기록한다.

실행 결과를 저장한 뒤 알려주면, 채택·미채택 이유를 포함한 16번 HTML 결과 보고서를 작성한다.